# Construcción de un meta clasificador para las 2 capas de análisis

In [39]:
#Montamos el notebook
from google.colab import drive
drive.mount('/content/drive')
LGBM_MODEL_PATH = '/content/drive/MyDrive/TFG_Posdata/models/lgbm_v1/model.pkl'
LGBM_CONFIG_PATH = '/content/drive/MyDrive/TFG_Posdata/models/lgbm_v1/config.json'
DISTILBERT_MODEL_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1'
DISTILBERT_TOKENIZER_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1'
DISTILBERT_CONFIG_PATH = '/content/drive/MyDrive/TFG_Posdata/models/distilbert_v1/own_config.json'
DATASET_PATH = '/content/drive/MyDrive/TFG_Posdata/scripts/datasets/dataset_final.csv'
MODEL_PATH = '/content/drive/MyDrive/TFG_Posdata/models/meta_v1/'

#Imports
import pandas as pd
import numpy as np
import sklearn
import json
import joblib
import torch
import transformers
import scipy
from sklearn.model_selection import train_test_split
import re
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
#Cargamos los 2 modelos clasificadores entrenados

lgbm_model = joblib.load(LGBM_MODEL_PATH)
lgbm_config = json.load(open(LGBM_CONFIG_PATH))

distilbert_model = transformers.AutoModelForSequenceClassification.from_pretrained(DISTILBERT_MODEL_PATH)
distilbert_tokenizer = transformers.AutoTokenizer.from_pretrained(DISTILBERT_TOKENIZER_PATH)
distilbert_config = json.load(open(DISTILBERT_CONFIG_PATH))

#Obtenemos el dataframe desde el CSV que contiene el dataset
df = pd.read_csv(DATASET_PATH)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [41]:
#----------------FUNCIÓN DE PREDICCIÓN CON DISTILBERT----------------#
def predict_distilbert(texts, batch_size=32):
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = distilbert_tokenizer(
            batch,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = distilbert_model(**inputs).logits

        probs = torch.nn.functional.softmax(logits, dim=1)
        all_probs.append(probs[:, 1].cpu().numpy())

    return np.concatenate(all_probs)

#----------------MARCADORES HEURÍSTICOS A CONTEMPLAR----------------#

#Presencia de palabras de urgencia
urgency_words = ['urgente','urgently','urgent','inmediatamente','immediately',
                   'ahora','now','hoy','today','bloquea','blocked','suspendida',
                   'suspended','cancel','cancela','verifique','verify']

#Presencia de palabras de acción
action_words = ['haga clic','click','acceda','access','llame','call','responda',
                   'reply','confirme','confirm','descargue','download','ingrese','enter']

#Presencia de términos financieros
financial_words = ['cuenta','account','banco','bank','tarjeta','card','pago','payment',
                   'transferencia','transfer','bizum','credito','credit','débito','debit']

#Presencia de términos relacionados con premios
prize_words = ['gratis','free','premio','prize','ganador','winner','regalo','gift',
                   'oferta','offer','descuento','discount','gana','win']

#Presencia de palabras de amenaza
threat_words = ['amenaza', 'threat', 'peligro', 'danger', 'dangerous', 'peligroso', 'cuidado',
                'beware', 'attention', 'atencion', 'careful', 'cuidado']

#Patrón de URLs
_http_pattern = r"https?://[^\s]+"
_www_pattern = r"www\.[^\s]+"
url_pattern     = re.compile(f"({_http_pattern})|({_www_pattern})")

#Patrón de acortadores de URL
shortener_pattern = re.compile(r'\b(bit\.ly|t\.co|tinyurl\.com|goo\.gl|ow\.ly|rb\.gy|cutt\.ly)\b')

#Patrón de número de teléfono
phone_pattern   = re.compile(r'(\+?[1-9]\d{1,14}|[0-9]{9,15})')

#----------------FUNCIÓN DE EXTRACCION DE MARCADORES HEURÍSTICOS----------------#
def extract_features (df):
  numeric_df = pd.DataFrame()

  #Características estructurales
  numeric_df['text_len'] = df['text'].str.len()
  numeric_df['word_count'] = df['text'].str.split().str.len()
  numeric_df['avg_word_len'] = numeric_df['text_len'] / numeric_df['word_count'].replace(0, 1)
  numeric_df['caps_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper())/max(len(x), 1))
  numeric_df['digit_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isdigit())/max(len(x), 1))
  numeric_df['excl_count'] = df['text'].str.count('!')
  numeric_df['ques_count'] = df['text'].str.count(r'\?')
  numeric_df['num_count'] = df['text'].str.count(r'\d')

  #características semánticas
  numeric_df['has_urgency'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in urgency_words))
  numeric_df['has_action'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in action_words))
  numeric_df['has_financial'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in financial_words))
  numeric_df['has_prize'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in prize_words))
  numeric_df['has_threat'] = df['text'].apply(lambda t: any(w in str(t).lower() for w in threat_words))

  #Características de contenido
  numeric_df['has_url'] = df['text'].apply(lambda t: bool(url_pattern.search(str(t))))
  numeric_df['has_phone'] = df['text'].apply(lambda t: bool(phone_pattern.search(str(t))))
  numeric_df['url_len'] = df['text'].apply(lambda t: (m := url_pattern.search(str(t))) and len(m.group(0)) or 0)
  numeric_df['has_shortener'] = df['text'].apply(lambda t: bool(shortener_pattern.search(str(t))))

  return numeric_df

In [42]:
#Obtenemos las entradas (X) y salidas (y) del dataset que utilizar para las predicciones con LBGM y DistilBERT
X_LBGM = extract_features(df)
X_DistilBERT = df['text']
y = (df['label'] == 'spam').astype(int)

#Dividimos el dataset en un conjunto de entrenamiento (train), uno de validación (val) y uno de prueba (test)
#para ambos DistilBERT y LGBM

X_DistilBERT_train, X_temp, y_train, y_temp = train_test_split(X_DistilBERT, y, test_size=0.4, random_state=42)
X_DistilBERT_val, X_DistilBERT_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

X_LBGM_train = X_LBGM.loc[X_DistilBERT_train.index]
X_LBGM_val   = X_LBGM.loc[X_DistilBERT_val.index]
X_LBGM_test  = X_LBGM.loc[X_DistilBERT_test.index]

In [43]:
#Movemos DistilBERT a GPU si es posible y lo ponemos en modo evaluación
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distilbert_model.to(device)
distilbert_model.eval()

#Generamos los scores sobre el conjunto de validación
scores_lgbm_val = lgbm_model.predict_proba(X_LBGM_val)[:, 1]
scores_distilbert_val = predict_distilbert(X_DistilBERT_val.tolist())
X_meta_train = np.column_stack((scores_lgbm_val, scores_distilbert_val))

#Generamos los scores sobre el conjunto de test
scores_lgbm_test = lgbm_model.predict_proba(X_LBGM_test)[:, 1]
scores_distilbert_test = predict_distilbert(X_DistilBERT_test.tolist())
X_meta_test = np.column_stack((scores_lgbm_test, scores_distilbert_test))

In [44]:
#Entrenamos el metaclasificador con los datos de validación
meta_model = LogisticRegression(class_weight='balanced', random_state=42)
meta_model.fit(X_meta_train, y_val)

LogisticRegression(class_weight='balanced', random_state=42)

In [45]:
#Evaluamos la eficacia del metaclasificador probándolo sobre el conjunto de prueba
y_pred_meta = meta_model.predict(X_meta_test)
y_scores_meta = meta_model.predict_proba(X_meta_test)[:, 1]
print(f"F1 = {f1_score(y_test, y_pred_meta, average='weighted'):.4f}")

F1 = 0.9845


In [47]:
#Guardamos el modelo y la configuración usada para entrenarlo

nombre_modelo = 'model.pkl'
with open(MODEL_PATH + nombre_modelo, 'wb') as f:
  joblib.dump(meta_model, f)
print("Modelo guardado con éxito")

nombre_config = 'config.json'
config = {
    "model_version": "meta_v1",
    "features": ["score_lgbm", "score_distilbert"],
    "threshold": 0.5
}
with open(MODEL_PATH + nombre_config, 'w') as f:
    json.dump(config, f)
print("Configuración guardada con éxito")

Modelo guardado con éxito
Configuración guardada con éxito
